EMBER2024 - APK 

In [1]:
import sys
print(sys.executable)
import thrember
import os
%pip install pandas
import pandas as pd
import numpy as np
import glob
import random
import shutil


/Users/angela/miniforge3/envs/ember/bin/python


/Users/angela/miniforge3/envs/ember/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
apk_directory = "/Users/angela/new_ember_apk"
small_directory = "/Users/angela/new_ember_apk_ss"

os.makedirs(apk_directory, exist_ok=True)
os.makedirs(small_directory, exist_ok=True)

thrember.download_dataset(apk_directory, file_type="APK")

Unzipping...


Unzipping...


In [9]:
nn_train = len(glob.glob(os.path.join(apk_directory, "*_APK_train.jsonl")))
nn_test = len(glob.glob(os.path.join(apk_directory, "*_APK_test.jsonl")))
print(nn_train, nn_test)
nnx_train = len(glob.glob(os.path.join(small_directory, "*_APK_train.jsonl")))
nnx_test = len(glob.glob(os.path.join(small_directory, "*_APK_test.jsonl")))
print(nn_train, nn_test)
print(nnx_train, nnx_test)


52 12
52 12
52 12


In [10]:
SEED = 42
per_week = 1000
randomseed = random.Random(SEED)
file = sorted(glob.glob(os.path.join(apk_directory, "*_APK_train.jsonl"))+
            glob.glob(os.path.join(apk_directory, "*_APK_test.jsonl"))) 
for fi in file:
    with open(fi) as apK:
        line = apK.readlines()
    keep = sorted(randomseed.sample(range(len(line)), min(per_week, len(line))))
    with open(os.path.join(small_directory, os.path.basename(fi)), "w") as out:
        out.writelines(line[xx] for xx in keep)
print("Sample of weekly files:", len(file))

for fi in glob.glob(os.path.join(apk_directory, "*_challenge_malicious.jsonl")):
    shutil.copy2(fi,small_directory)

print(len(glob.glob(os.path.join(small_directory, "*.jsonl"))))

Sample of weekly files: 64
128


In [11]:
part = sorted(f for f in os.listdir(small_directory) if not f.endswith(".jsonl"))
print(part)
for f in part:
    os.remove(os.path.join(small_directory,f))
print(len(os.listdir(small_directory)))

[]
128


In [12]:
thrember.create_vectorized_features(small_directory)

Preparing to vectorize raw features
Vectorizing training set


100%|██████████| 52000/52000 [12:00<00:00, 72.16it/s] 


Vectorizing test set


100%|██████████| 12000/12000 [02:51<00:00, 70.15it/s] 


Vectorizing challenge set


100%|██████████| 6315/6315 [01:30<00:00, 70.16it/s] 


In [13]:
X_train_APK, y_train_APK = thrember.read_vectorized_features(small_directory, subset="train")
X_test_APK, y_test_APK = thrember.read_vectorized_features(small_directory, subset="test")
print(X_train_APK.shape, y_train_APK.shape)
print(X_test_APK.shape, y_test_APK.shape)

print("Mal fract:", y_train_APK.mean())
print(sorted(f_APK for f_APK in os.listdir(small_directory) if not f_APK.endswith(".jsonl")))


(52000, 2568) (52000,)
(12000, 2568) (12000,)
Mal fract: 0.49884615384615383
['X_challenge.dat', 'X_test.dat', 'X_train.dat', 'y_challenge.dat', 'y_test.dat', 'y_train.dat']


In [15]:
# time splitting 80/40
training_e = 41600
X_train_APK_6we = X_train_APK[:training_e]
y_train_APK_6we = y_train_APK[:training_e]
x_validation_apk = X_train_APK[:training_e]
y_validation_apk = y_train_APK[:training_e]
# testing split 
X_test_APK_Final = X_test_APK
y_test_APK_final = y_test_APK
print("Split: ", X_train_APK_6we.shape, y_train_APK_6we.shape)
print("Validation:", x_validation_apk.shape, y_validation_apk.shape)
print("Testing split:", X_test_APK_Final.shape, y_test_APK_final.shape)


Split:  (41600, 2568) (41600,)
Validation: (41600, 2568) (41600,)
Testing split: (12000, 2568) (12000,)


In [17]:
import lightgbm as lgb
from sklearn.metrics import precision_score,recall_score, roc_auc_score, average_precision_score
model_APK = lgb.LGBMClassifier(n_estimators=200, random_state=42)
model_APK.fit(X_train_APK_6we, y_train_APK_6we)

proba_APK = model_APK.predict_proba(X_test_APK)[:,1]
prediction_APK= (proba_APK >= 0.5).astype(int)

print("precision: ", precision_score(y_test_APK, prediction_APK))
print("recall: ", recall_score(y_test_APK, prediction_APK))
print("auroc: ", roc_auc_score(y_test_APK, proba_APK))
print("ap ", average_precision_score(y_test_APK, proba_APK))

[LightGBM] [Info] Number of positive: 20717, number of negative: 20883
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.262642 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 169995
[LightGBM] [Info] Number of data points in the train set: 41600, number of used features: 686
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.498005 -> initscore=-0.007981
[LightGBM] [Info] Start training from score -0.007981
precision:  0.9270417422867514
recall:  0.8355962702437428
auroc:  0.9588302584324979
ap  0.9620917272753273


In [18]:
# saving it
test_row_ID_APK = [
    f"EMBER_APK_TEST_{m}"
    for m in range(len(y_test_APK_final))

]
baseline_APK= pd.DataFrame({
    "row_id": test_row_ID_APK,
    "value": "baseline",
    "column": "APK_LightGBM",
    "score": proba_APK,
    "threshold": 0.5,
    "label": y_test_APK_final
})
baseline_APK.to_csv("Baseline_APK.csv", index=False)